## Simple Data Preprocessing Automation

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time

In [24]:
# ==========================================
# [New] Anthropic News Scraper (Date Range Version)
# ==========================================

import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time

def parse_anthropic_date(date_str):
    """Parse dates like 'Dec 19, 2025' or 'Jan 3, 2026'"""
    if not date_str: return None
    try:
        return datetime.strptime(date_str.strip(), '%b %d, %Y')
    except ValueError:
        try:
            return datetime.strptime(date_str.strip().split('\n')[0], '%b %d, %Y')
        except:
            return None

def normalize_date(d):
    """Convert string 'YYYY-MM-DD' to datetime, or return datetime as is."""
    if isinstance(d, str):
        return datetime.strptime(d, '%Y-%m-%d')
    return d

def get_anthropic_news_list(start_date, end_date=None):
    """
    Scrapes the list of articles from Anthropic News.
    Filters articles published between `start_date` and `end_date` (inclusive).
    start_date, end_date: 'YYYY-MM-DD' strings or datetime objects.
    If end_date is None, defaults to today.
    """
    url = "https://www.anthropic.com/news"
    
    # Normalize dates
    start_dt = normalize_date(start_date)
    end_dt = normalize_date(end_date) if end_date else datetime.now()
    
    # Ensure we include the whole end day by setting time to 23:59:59 if needed,
    # but usually simple comparison works if we strip time from article date.
    # Let's strip time from all for safe comparison
    start_dt = start_dt.replace(hour=0, minute=0, second=0, microsecond=0)
    end_dt = end_dt.replace(hour=23, minute=59, second=59, microsecond=999999)
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to fetch list page: {response.status_code}")
            return []
        
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        # Robust Search
        all_links = soup.find_all('a', href=True)
        candidate_items = [a for a in all_links if a.find('time')]
        
        print(f"Found {len(candidate_items)} potential article cards. Filtering range: {start_dt.strftime('%Y-%m-%d')} ~ {end_dt.strftime('%Y-%m-%d')}")
        
        for item in candidate_items:
            try:
                href = item.get('href')
                full_url = f"https://www.anthropic.com{href}" if href.startswith('/') else href
                
                time_tag = item.find('time')
                date_str = time_tag.get_text(strip=True) if time_tag else ""
                article_date = parse_anthropic_date(date_str)
                
                full_card_text = item.get_text(" ", strip=True)
                if date_str:
                    title = full_card_text.replace(date_str, "").strip()
                else:
                    title = full_card_text
                
                if article_date:
                    # Date Range Check
                    # Normalize article date (it usually comes with 00:00:00)
                    if start_dt <= article_date <= end_dt:
                        articles.append({
                            'url': full_url,
                            'title': title,
                            'date': date_str
                        })
                    
            except Exception as e:
                print(f"Error parsing list item: {e}")
                continue
                
        return articles
    except Exception as e:
        print(f"Error in get_anthropic_news_list: {e}")
        return []

def scrape_anthropic_article_detail(url):
    """
    Scrapes detail of a single Anthropic article.
    Returns { 'full_text', 'url', 'date' }
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to fetch article: {url}")
            return None
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        content_div = soup.find('div', class_=lambda c: c and 'Body-module' in str(c))
        if not content_div: content_div = soup.find('article')
        if not content_div: content_div = soup.find('main')
        
        if content_div:
            for tag in content_div(['script', 'style', 'button', 'nav', 'footer', 'form']):
                tag.decompose()
            for h in content_div.find_all('header'): h.decompose()
            full_text = content_div.get_text(separator='\n', strip=True)
        else:
            full_text = soup.body.get_text(separator='\n', strip=True) if soup.body else ""
        
        date_str = "Unknown"
        time_tag = soup.find('time')
        if time_tag:
            date_str = time_tag.get_text(strip=True)

        return {
            "raw_content": full_text,
            "source_url": url,
            "date": date_str
        }

    except Exception as e:
        print(f"Error scraping article {url}: {e}")
        return None

def run_anthropic_scraper(start_date, end_date=None):
    """
    Executes the scraper for the given date range.
    Example: run_anthropic_scraper('2025-12-19', '2025-12-31')
    """
    if end_date is None:
        end_date = datetime.now().strftime('%Y-%m-%d')
        
    print(f"Starting Anthropic Scraper ({start_date} ~ {end_date})...")
    target_articles = get_anthropic_news_list(start_date, end_date)
    
    results = []
    print(f"Processing {len(target_articles)} articles...")
    
    for article in target_articles:
        print(f"  > Scraping: {article['title'][:30]}... ({article['date']})")
        details = scrape_anthropic_article_detail(article['url'])
        if details:
            if (not details['date'] or details['date'] == "Unknown") and article['date']:
                details['date'] = article['date']
            
            # Merge Title from List
            details['raw_title'] = article['title']
            details['source_name'] = "Anthropic"
            
            results.append(details)
        time.sleep(1)
        
    print(f"Done. Scraped {len(results)} articles.")
    return results


In [26]:
# ==========================================
# [Test] Scrape with Start/End Date
# ==========================================

# Example: Scrape from Dec 19, 2025 to Today
start_input = "2025-12-19"
end_input = "2025-12-31" # Or set to None for today

if 'run_anthropic_scraper' in globals():
    results = run_anthropic_scraper(start_date=start_input, end_date=end_input)
    
    print(f"\n[Result Check]")
    for item in results:
        print(f"- [{item['date']}] {item['raw_title']}")
else:
    print("Please run the scraper definition cell first.")


Starting Anthropic Scraper (2025-12-19 ~ 2025-12-31)...
Found 15 potential article cards. Filtering range: 2025-12-19 ~ 2025-12-31
Processing 1 articles...
  > Scraping: Policy Sharing our compliance ... (Dec 19, 2025)
Done. Scraped 1 articles.

[Result Check]
- [Dec 19, 2025] Policy Sharing our compliance framework for California's Transparency in Frontier AI Act


In [27]:
item

{'raw_content': "On January 1, California's Transparency in Frontier AI Act (\nSB 53\n) will go into effect. It establishes the nation’s first frontier AI safety and transparency requirements for catastrophic risks.\nWhile we have long advocated for a federal framework, Anthropic\nendorsed\nSB 53 because we believe frontier AI developers like ourselves should be transparent about how they assess and manage these risks. Importantly, the law balances the need for strong safety practices, incident reporting, and whistleblower protections—while preserving flexibility in how developers implement their safety measures, and exempting smaller companies from unnecessary regulatory burdens.\nOne of the law’s key requirements is that frontier AI developers publish a framework describing how they assess and manage catastrophic risks. Our Frontier Compliance Framework (FCF) is now available to the public,\nhere\n. Below, we discuss what’s included within it, and highlight what we think should come 